# Proyecto 2: Fine-tuning para tareas de PLN

Procesamiento de Lenguaje Natural, Universidad de Antioquia.

Fine-tuning de modelos de Hugging Face para tres tareas: clasificación de secuencias, etiquetado de secuencias y sequence-to-sequence.

**Cómo correrlo**
- **Colab:** Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4). El token de Hugging Face va en los Secrets de Colab con el nombre `HF_TOKEN`. Los checkpoints se guardan en Google Drive.
- **Local:** `uv sync` en la raíz del repo, kernel `proyecto2-nlp` y el token en el archivo `.env`.

## 0. Configuración del entorno

La primera celda detecta si el notebook corre en Colab o en local. En Colab instala lo que falta, con las mismas versiones del `uv.lock` del repo, y **no toca torch** (se usa el que trae Colab con CUDA). En local no instala nada: las dependencias salen de `uv sync`.

In [ ]:
# Detección del entorno e instalación (solo en Colab)
import importlib.metadata as md
import os
import re
import subprocess
import sys
import tempfile

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

# Mismas versiones del uv.lock. Si se actualiza el lock hay que cambiarlas aquí;
# scripts/check_env.py avisa cuando no coinciden.
COLAB_PINS = {
    "transformers": "5.23.0",
    "tokenizers": "0.23.3",
    "huggingface-hub": "2.4.0",
    "datasets": "5.2.0",
    "evaluate": "0.4.6",
    "accelerate": "1.16.0",
    "seqeval": "1.2.2",
    "sacrebleu": "2.6.0",
    "rouge-score": "0.1.2",
}


def _norm(nombre):
    return re.sub(r"[-_.]+", "-", nombre).lower()


def _instaladas():
    return {_norm(d.metadata["Name"]): d.version for d in md.distributions()}


if IN_COLAB:
    antes = _instaladas()
    pendientes = [f"{p}=={v}" for p, v in COLAB_PINS.items() if antes.get(_norm(p)) != v]
    assert "torch" in antes, "Colab debería traer torch preinstalado"
    if pendientes:
        # El constraint deja torch exactamente como viene en Colab: si algo pidiera
        # otra versión, pip falla en vez de reinstalarlo.
        with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
            f.write(f"torch=={antes['torch']}\n")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-c", f.name, *pendientes],
            check=True,
        )
        despues = _instaladas()
        assert despues["torch"] == antes["torch"], "pip cambió la versión de torch"
        cambiadas = {p for p, v in despues.items() if antes.get(p) != v}
        # Si alguna librería actualizada ya estaba importada, hay que reiniciar la sesión
        cargadas = sorted(
            mod
            for mod, dists in md.packages_distributions().items()
            if mod in sys.modules and any(_norm(d) in cambiadas for d in dists)
        )
        if cargadas:
            raise SystemExit(
                f"Se actualizaron librerías que ya estaban cargadas ({', '.join(cargadas)}). "
                "Entorno de ejecución > Reiniciar sesión y vuelve a correr el notebook desde arriba."
            )
    for p in ["torch", *COLAB_PINS]:
        print(f"{p:<16} {md.version(p)}")
else:
    print("Entorno local: las dependencias vienen de uv.lock (uv sync). No se instala nada.")

In [ ]:
# Configuración general: Drive, token de Hugging Face, dispositivo, semilla y rutas
import platform
from pathlib import Path

# En Mac, las operaciones que MPS no soporta caen a CPU en vez de fallar.
# Tiene que definirse antes de importar torch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch
import transformers
import datasets
from huggingface_hub import login

SEED = 42

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/proyecto2-nlp/outputs")
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:  # el secret no existe o el notebook no tiene acceso
        HF_TOKEN = None
        print(f"No se pudo leer HF_TOKEN de los Secrets de Colab ({type(e).__name__}).")
else:
    from dotenv import load_dotenv

    # Raíz del repo: la carpeta que tiene pyproject.toml (el notebook vive en notebooks/)
    PROJECT_ROOT = next(
        (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
        Path.cwd(),
    )
    load_dotenv(PROJECT_ROOT / ".env")
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    HF_TOKEN = os.getenv("HF_TOKEN") or None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Login en Hugging Face sin mostrar el token
if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Login en Hugging Face: OK")
    except Exception as e:
        print(f"Login en Hugging Face falló ({type(e).__name__}). Revisa que el token sea válido.")
else:
    print("Sin HF_TOKEN: se sigue sin login (los modelos y datasets públicos funcionan igual).")

# Dispositivo
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    detalle = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    detalle = "Apple Silicon (MPS)"
else:
    DEVICE = torch.device("cpu")
    detalle = platform.processor() or "CPU"
if IN_COLAB and DEVICE.type != "cuda":
    print("OJO: no hay GPU. Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU.")

# Semilla (random, numpy y torch)
transformers.set_seed(SEED)

print(f"Entorno:      {'Colab' if IN_COLAB else 'local'}")
print(f"Python:       {platform.python_version()}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"Dispositivo:  {DEVICE} ({detalle})")
print(f"Semilla:      {SEED}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

## Introducción

## Tarea 1: Clasificación de secuencias

### 1.1 Descripción del dataset

### 1.2 Análisis exploratorio (EDA)

### 1.3 Búsqueda de hiperparámetros

### 1.4 Entrenamiento con la data completa

### 1.5 Evaluación del mejor modelo

## Tarea 2: Etiquetado de secuencias

### 2.1 Descripción del dataset

### 2.2 Análisis exploratorio (EDA)

### 2.3 Búsqueda de hiperparámetros

### 2.4 Entrenamiento con la data completa

### 2.5 Evaluación del mejor modelo

## Tarea 3: Sequence-to-sequence

### 3.1 Descripción del dataset

### 3.2 Análisis exploratorio (EDA)

### 3.3 Búsqueda de hiperparámetros

### 3.4 Entrenamiento con la data completa

### 3.5 Evaluación del mejor modelo

## Conclusiones